# 01 — Setup & Data Prep

**한 번만 실행**: 환경 설정 + Recipe1MSubs 다운로드 + 그래프팀 파일 변환 + (선택) smoke test.
데이터가 Drive에 이미 변환되어 있으면 셀 1-6만 돌려도 됨 (NUM_TOTAL, NUM_ING 확인용).

이후 학습 노트북 (02, 03, 04) 은 각각 별도 Colab 세션에서 돌리면 됨.

## 1. 환경 설정

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q torch_geometric pandas numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
PROJECT_ROOT = '/content/drive/MyDrive/CS471_project'
CODE_DIR = f'{PROJECT_ROOT}/code'
DATA_RAW = f'{PROJECT_ROOT}/data_raw'
DATA_DIR = f'{PROJECT_ROOT}/data'
OUTPUT_DIR = f'{PROJECT_ROOT}/outputs'

os.makedirs(DATA_RAW, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(CODE_DIR)

print(f'Working dir: {os.getcwd()}')
!ls

In [ ]:
import torch, torch_geometric
print(f'PyTorch: {torch.__version__}')
print(f'PyG    : {torch_geometric.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}  ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu"})')

## 2. 데이터 준비

Recipe1MSubs (4 pkl) 자동 다운로드 + 그래프팀 파일 존재 확인.

In [ ]:
# Recipe1MSubs (저자 공식 release)
import os
PKLS = [
    'train_comments_subs.pkl',
    'val_comments_subs.pkl',
    'test_comments_subs.pkl',
    'vocab_ingrs.pkl',
]
for pkl in PKLS:
    path = f'{DATA_RAW}/{pkl}'
    if os.path.exists(path):
        print(f'✓ {pkl} already exists ({os.path.getsize(path) // 1024} KB)')
    else:
        url = f'https://dl.fbaipublicfiles.com/gismo/{pkl}'
        !wget -q -O {path} {url}
        print(f'↓ downloaded {pkl}')

In [ ]:
# 그래프팀 파일 체크
REQUIRED = ['flavorgraph_edges.csv', 'nodes_filtered.csv', 'usda_mapping.json']
missing = [f for f in REQUIRED if not os.path.exists(f'{DATA_RAW}/{f}')]
if missing:
    print(f'✗ Missing in {DATA_RAW}:')
    for f in missing:
        print(f'  - {f}')
    print(f'\nUpload these to {DATA_RAW} and re-run this cell.')
else:
    print(f'✓ All graph team files present in {DATA_RAW}')
    !ls -lh {DATA_RAW}

In [ ]:
# 변환 실행 — pkl + 그래프팀 파일 → 우리 모델 형식
!python convert_data.py --data_raw {DATA_RAW} --out_dir {DATA_DIR}

In [ ]:
# 변환 결과 통계 + 추후 train cell이 사용할 값들 로드
import json
with open(f'{DATA_DIR}/data_meta.json') as f:
    META = json.load(f)

NUM_TOTAL = META['num_total_nodes']
NUM_ING   = META['num_ingredients']

print(f'\n=== Data Summary ===')
for k, v in META.items():
    print(f'  {k}: {v}')
print(f'\nWill train with:')
print(f'  --num_total_nodes {NUM_TOTAL}')
print(f'  --num_ingredients {NUM_ING}')

## 3. (선택) Smoke test — 코드 빠른 sanity check

Mock data로 1분 안에 baseline + mvp가 학습/평가 사이클 한 번 돌리는지 확인. 시간 없으면 skip하고 02부터 실행해도 됨.

In [ ]:
# Mock data 생성 (F/D 노드 포함)
!python mock_data.py --out_dir ./mock_data \
    --num_ingredients 200 --num_flavor_compounds 50 \
    --num_recipes 500 \
    --num_pairs_train 2000 --num_pairs_val 300 --num_pairs_test 300

# Baseline smoke (mock)
!python train_v1.py --mode baseline --data_dir ./mock_data --output_dir ./mock_out \
    --num_total_nodes 250 --num_ingredients 200 \
    --embed_dim 64 --batch_size 32 \
    --max_epochs 3 --patience 5 --num_workers 0

# MVP smoke (mock)
!python train_v1.py --mode mvp --data_dir ./mock_data --output_dir ./mock_out \
    --num_total_nodes 250 --num_ingredients 200 \
    --embed_dim 64 --batch_size 32 \
    --max_epochs 3 --patience 5 --num_workers 0 \
    --test_g_overrides auto 1_0 0_1

print('\n=== Smoke test passed if both runs completed without error ===')